# 01 — CICIDS2017 Data Exploration & Schema Audit
**SMART IDS v3** | Network Flow Telemetry Inspection & Pathology Quantification

This notebook loads either the raw CICIDS2017 CSV exports from `data/raw/` or the explicitly labeled synthetic demonstration dataset when `data/raw/` is empty, and audits column schemas, class imbalance ratios, `Inf`/`NaN` artifacts, and top-variance feature correlations.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_app_config
from src.data.loader import load_cicids2017_corpus

cfg = get_app_config()
df_raw, source_info = load_cicids2017_corpus(
    raw_dir=cfg.paths.resolve('raw_data_dir'),
    synthetic_dir=cfg.paths.resolve('synthetic_data_dir'),
    data_cfg=cfg.data,
    seed=cfg.project.random_seed,
)
print('Source Type:', source_info.source_type, '| Synthetic:', source_info.is_synthetic)
print('Shape:', df_raw.shape)
df_raw.head()

In [ ]:
label_counts = df_raw[cfg.data.label_column].value_counts()
print('Class Distribution:\n', label_counts)

num_df = df_raw.select_dtypes(include=[np.number])
inf_count = int(np.isinf(num_df.to_numpy(dtype=np.float64, copy=False)).sum())
nan_count = int(df_raw.isna().sum().sum())
dup_count = int(df_raw.duplicated().sum())
print(f'Pathologies -> Inf: {inf_count:,} | NaN: {nan_count:,} | Duplicates: {dup_count:,}')

fig, ax = plt.subplots(figsize=(10, 4.5))
label_counts.sort_values().plot(kind='barh', ax=ax, color='#0ea5e9')
ax.set_title(f'Class Distribution ({source_info.source_type})')
ax.set_xlabel('Flow Count')
plt.tight_layout()
plt.show()